# 🔭 Bayesian Brightness Inference
## YZM212 Makine Öğrenmesi – 4. Laboratuvar Ödevi

**Konu:** Uzak Bir Galaksinin Parlaklık Analizi  
**Yöntem:** MCMC tabanlı Bayesyen Çıkarım (`emcee`)  

---

### Amaç

Gürültülü gözlem verilerinden bir gök cisminin:
- **μ (mu):** Gerçek parlaklığını
- **σ (sigma):** Ölçüm belirsizliğini

Bayesyen yöntemlerle posterior dağılım biçiminde tahmin etmek.

### Bayes Teoremi

$$P(\theta \mid D) = \frac{P(D \mid \theta)\, P(\theta)}{P(D)}$$

| Bileşen | Anlamı |
|---|---|
| $P(\theta \mid D)$ — Posterior | Veriyi gördükten sonra parametreler hakkında ne biliyoruz |
| $P(D \mid \theta)$ — Likelihood | Bu parametreler doğruysa veriyi gözlemleme olasılığı |
| $P(\theta)$ — Prior | Veriyi görmeden önce parametreler hakkındaki ön bilgimiz |
| $P(D)$ — Evidence | Normalizasyon sabiti (MCMC'de hesaplanmaz) |

## Bölüm 1 – Kütüphane ve Kurulum

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import emcee
import corner
from scipy.stats import norm

# Grafik kalitesi
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Çıktı klasörü
os.makedirs('figures', exist_ok=True)

print('Kütüphaneler yüklendi.')
print(f'emcee versiyonu: {emcee.__version__}')

## Bölüm 2 – Sentetik Veri Üretimi

Gerçekte **bilinmediğini varsaydığımız** ama kontrol değişkeni olarak bildiğimiz parametreler:

In [ ]:
# --- Sabitler ---
TRUE_MU    = 150.0   # Gerçek parlaklık
TRUE_SIGMA = 10.0    # Gerçek ölçüm gürültüsü
N_OBS      = 50      # Gözlem sayısı
SEED       = 42

# --- Veri üretimi ---
np.random.seed(SEED)
data = TRUE_MU + TRUE_SIGMA * np.random.randn(N_OBS)

print(f'Gözlem sayısı : {N_OBS}')
print(f'Örnek ortalama: {data.mean():.4f}  (Gerçek μ = {TRUE_MU})')
print(f'Örnek std     : {data.std(ddof=1):.4f}  (Gerçek σ = {TRUE_SIGMA})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(data, bins=12, density=True, alpha=0.6, color='steelblue',
        edgecolor='white', label='Gürültülü Gözlemler')

x = np.linspace(data.min() - 5, data.max() + 5, 300)
ax.plot(x, norm.pdf(x, TRUE_MU, TRUE_SIGMA),
        'r-', lw=2, label=f'Gerçek Dağılım N({TRUE_MU}, {TRUE_SIGMA}²)')
ax.axvline(TRUE_MU, color='red', linestyle='--', alpha=0.6, label=f'μ={TRUE_MU}')

ax.set_xlabel('Parlaklık (Flux Birimi)')
ax.set_ylabel('Olasılık Yoğunluğu')
ax.set_title('Sentetik Gözlem Verisi')
ax.legend()
plt.tight_layout()
plt.savefig('figures/01_synthetic_data.png', dpi=150)
plt.show()

## Bölüm 3 – Bayesyen Model Fonksiyonları

### Gaussian Log-Likelihood

$$\ln P(D \mid \mu, \sigma) = -\frac{1}{2} \sum_{i=1}^{n} \left[ \left(\frac{x_i - \mu}{\sigma}\right)^2 + \ln(2\pi\sigma^2) \right]$$

### Uniform Prior

$$P(\mu, \sigma) = \begin{cases} 1 & 0 < \mu < 300, \; 0 < \sigma < 50 \\ 0 & \text{aksi halde} \end{cases}$$

In [ ]:
def log_likelihood(theta, obs):
    """Gaussian log-likelihood. sigma <= 0 fiziksel değildir."""
    mu, sigma = theta
    if sigma <= 0:
        return -np.inf
    return -0.5 * np.sum(((obs - mu) / sigma)**2 + np.log(2.0 * np.pi * sigma**2))


def log_prior(theta, mu_low=0.0, mu_high=300.0, sigma_high=50.0):
    """Düz (uniform) prior."""
    mu, sigma = theta
    if mu_low < mu < mu_high and 0.0 < sigma < sigma_high:
        return 0.0
    return -np.inf


def log_probability(theta, obs, mu_low=0.0, mu_high=300.0, sigma_high=50.0):
    """Log-posterior = log-prior + log-likelihood."""
    lp = log_prior(theta, mu_low, mu_high, sigma_high)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, obs)


print('Model fonksiyonları tanımlandı.')

## Bölüm 4 – MCMC Örneklemesi

| MCMC Parametresi | Değer | Açıklama |
|---|---|---|
| Başlangıç | [140, 5] | Gerçek değerden uzak başlatıldı |
| Walker sayısı | 32 | Paralel zincir sayısı |
| Toplam adım | 2000 | Örnekleme uzunluğu |
| Burn-in | 500 | Atılan başlangıç adımları |
| Thinning | 15 | Otokorelasyon azaltma |

In [ ]:
N_WALKERS = 32
N_STEPS   = 2000
BURN_IN   = 500
THIN      = 15
NDIM      = 2

np.random.seed(SEED)
initial = np.array([140.0, 5.0])
pos = initial + 1e-4 * np.random.randn(N_WALKERS, NDIM)

sampler = emcee.EnsembleSampler(
    N_WALKERS, NDIM, log_probability, args=(data,)
)
sampler.run_mcmc(pos, N_STEPS, progress=True)

flat_samples = sampler.get_chain(discard=BURN_IN, thin=THIN, flat=True)
print(f'\nÖrneklem boyutu: {flat_samples.shape[0]} (bağımsız MCMC örneği)')

## Bölüm 5 – Trace Plot (Walker Zincirleri)

In [ ]:
chain = sampler.get_chain()
labels = [r'$\mu$ (Parlaklık)', r'$\sigma$ (Gürültü)']

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(chain[:, :, i], alpha=0.35, lw=0.6, color='steelblue')
    ax.axvline(BURN_IN, color='red', linestyle='--', lw=1.5,
               label='Burn-in sınırı')
    ax.set_ylabel(label, fontsize=12)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Adım (MCMC İterasyonu)')
fig.suptitle('MCMC Trace Plot – Walker Zincirleri', fontsize=13)
plt.tight_layout()
plt.savefig('figures/02_trace_base.png', dpi=150)
plt.show()

> **Trace Plot Yorumu:** Burn-in süresi boyunca walker'lar başlangıç noktasından uzaklaşıyor. ~500 adım sonrasında zincirler sabit bir dağılım etrafında salınıyor — bu, yakınsama işaretidir.

## Bölüm 6 – Posterior İstatistikleri ve Sonuç Tablosu

In [ ]:
mu_med  = np.median(flat_samples[:, 0])
mu_lo   = np.percentile(flat_samples[:, 0], 16)
mu_hi   = np.percentile(flat_samples[:, 0], 84)

sig_med = np.median(flat_samples[:, 1])
sig_lo  = np.percentile(flat_samples[:, 1], 16)
sig_hi  = np.percentile(flat_samples[:, 1], 84)

print('=' * 65)
print('  TEMEL SENARYO  (n=50, geniş prior)  –  Posterior Sonuçları')
print('=' * 65)
print(f'  {"Parametre":<14} {"Gerçek":>8} {"Median":>10} {"  %16":>8} {"  %84":>8} {"  |Hata|":>10}')
print('-' * 65)
print(f'  {"μ (Parlaklık)":<14} {TRUE_MU:>8.2f} {mu_med:>10.4f} {mu_lo:>8.4f} {mu_hi:>8.4f} {abs(mu_med - TRUE_MU):>10.4f}')
print(f'  {"σ (Gürültü)":<14} {TRUE_SIGMA:>8.2f} {sig_med:>10.4f} {sig_lo:>8.4f} {sig_hi:>8.4f} {abs(sig_med - TRUE_SIGMA):>10.4f}')
print('=' * 65)
print(f'  μ  CI genişliği (%68): {mu_hi - mu_lo:.4f}')
print(f'  σ  CI genişliği (%68): {sig_hi - sig_lo:.4f}')

## Bölüm 7 – Corner Plot (Temel Senaryo)

In [ ]:
fig = corner.corner(
    flat_samples,
    labels=[r'$\mu$ (Parlaklık)', r'$\sigma$ (Gürültü)'],
    truths=[TRUE_MU, TRUE_SIGMA],
    truth_color='red',
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 11},
    color='steelblue'
)
fig.suptitle('Corner Plot – Temel Senaryo (n=50)', fontsize=13, y=1.02)
plt.savefig('figures/03_corner_base.png', dpi=150, bbox_inches='tight')
plt.show()

> **Corner Plot Yorumu:**
> - Diyagonal: Her parametrenin marjinal posterior dağılımı. μ simetrik, σ hafif sağa-çarpık.
> - Çapraz panel: μ ve σ arasındaki ilişki. Elipsin dikeyeye yakın durması, parametrelerin büyük ölçüde **bağımsız** olduğunu gösterir.
> - Kırmızı çizgiler: Gerçek değerlerin posterior yoğunluğunun içinde kaldığı görülmektedir — model başarılı.

---

## Bölüm 8 – Analiz Soruları

### 8.1 Prior Etkisi – Dar Prior (100 < μ < 110)

In [ ]:
np.random.seed(SEED)
pos_narrow = np.array([105.0, 5.0]) + 1e-4 * np.random.randn(N_WALKERS, NDIM)

sampler_narrow = emcee.EnsembleSampler(
    N_WALKERS, NDIM, log_probability,
    args=(data, 100.0, 110.0, 50.0)
)
sampler_narrow.run_mcmc(pos_narrow, N_STEPS, progress=True)
fs_narrow = sampler_narrow.get_chain(discard=BURN_IN, thin=THIN, flat=True)

mu_n_med  = np.median(fs_narrow[:, 0])
sig_n_med = np.median(fs_narrow[:, 1])
sig_n_lo  = np.percentile(fs_narrow[:, 1], 16)
sig_n_hi  = np.percentile(fs_narrow[:, 1], 84)

print('=' * 60)
print('  DAR PRIOR SENARYOSU  (100 < μ < 110)')
print('=' * 60)
print(f'  μ  Median: {mu_n_med:.4f}   Hata: {abs(mu_n_med - TRUE_MU):.4f}')
print(f'  σ  Median: {sig_n_med:.4f}   Hata: {abs(sig_n_med - TRUE_SIGMA):.4f}')
print(f'  σ  Güven aralığı: [{sig_n_lo:.2f}, {sig_n_hi:.2f}]')
print('=' * 60)

In [ ]:
fig = corner.corner(
    fs_narrow,
    labels=[r'$\mu$ (Parlaklık)', r'$\sigma$ (Gürültü)'],
    truths=[TRUE_MU, TRUE_SIGMA],
    truth_color='red',
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    color='darkorange'
)
fig.suptitle('Corner Plot – Dar Prior (100 < μ < 110)', fontsize=13, y=1.02)
plt.savefig('figures/04_corner_narrow_prior.png', dpi=150, bbox_inches='tight')
plt.show()

> **Dar Prior Yorumu:**
>
> Gerçek μ=150, prior aralığı (100, 110)'un tamamen dışındadır. MCMC bu bölgeye hiç erişemez. Sonuç olarak:
> - μ tahmini ~109'a saplanır (prior üst sınırına yakın)
> - σ, gözlem-model uyumsuzluğunu "veri çok gürültülüymüş" diye yorumlayarak ~40'a fırlar
>
> **Ders:** Gerçekçi olmayan bir prior, veriyi yanlış yorumlar ve tüm çıkarımı bozar.

---

### 8.2 Veri Miktarı Etkisi – n=5

In [ ]:
np.random.seed(SEED)
data5 = TRUE_MU + TRUE_SIGMA * np.random.randn(5)

np.random.seed(SEED)
pos5 = initial + 1e-4 * np.random.randn(N_WALKERS, NDIM)
sampler5 = emcee.EnsembleSampler(N_WALKERS, NDIM, log_probability, args=(data5,))
sampler5.run_mcmc(pos5, N_STEPS, progress=True)
fs5 = sampler5.get_chain(discard=BURN_IN, thin=THIN, flat=True)

mu5_med  = np.median(fs5[:, 0])
mu5_lo   = np.percentile(fs5[:, 0], 16)
mu5_hi   = np.percentile(fs5[:, 0], 84)
sig5_med = np.median(fs5[:, 1])
sig5_lo  = np.percentile(fs5[:, 1], 16)
sig5_hi  = np.percentile(fs5[:, 1], 84)

print('=' * 65)
print('  KARŞILAŞTIRMA: n=50  vs  n=5')
print('=' * 65)
print(f'  {"":<20} {"n=50":>12} {"n=5":>12}')
print(f'  {"-"*44}')
print(f'  {"μ median":<20} {mu_med:>12.4f} {mu5_med:>12.4f}')
print(f'  {"μ CI genişliği":<20} {mu_hi - mu_lo:>12.4f} {mu5_hi - mu5_lo:>12.4f}')
print(f'  {"σ median":<20} {sig_med:>12.4f} {sig5_med:>12.4f}')
print(f'  {"σ CI genişliği":<20} {sig_hi - sig_lo:>12.4f} {sig5_hi - sig5_lo:>12.4f}')
print('=' * 65)
print(f'  μ belirsizliği {(mu5_hi - mu5_lo)/(mu_hi - mu_lo):.1f}× arttı')
print(f'  σ belirsizliği {(sig5_hi - sig5_lo)/(sig_hi - sig_lo):.1f}× arttı')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
param_names = [r'$\mu$ (Parlaklık)', r'$\sigma$ (Gürültü)']
true_vals   = [TRUE_MU, TRUE_SIGMA]

for i, (ax, name, tv) in enumerate(zip(axes, param_names, true_vals)):
    ax.hist(flat_samples[:, i], bins=40, density=True,
            alpha=0.55, color='steelblue', label='n=50')
    ax.hist(fs5[:, i], bins=40, density=True,
            alpha=0.55, color='tomato', label='n=5')
    ax.axvline(tv, color='black', linestyle='--', lw=1.5,
               label=f'Gerçek = {tv}')
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('Olasılık Yoğunluğu')
    ax.set_title(f'Posterior: {name}')
    ax.legend(fontsize=9)

fig.suptitle('Posterior Karşılaştırması: n=50 vs n=5', fontsize=13)
plt.tight_layout()
plt.savefig('figures/05_posterior_comparison_n5_vs_n50.png', dpi=150)
plt.show()

> **Veri Miktarı Yorumu:**
>
> n=5 ile n=50 arasında μ belirsizliği **3.1×**, σ belirsizliği **4.5×** arttı.
>
> İstatistiksel gerekçe: Ortalama SE = σ/√n. n=50'den n=5'e geçince √(50/5) ≈ **3.16×** genişleme beklenir — tam uyum.
> σ tahmini daha az veri ile orantısız biçimde kötüleşir çünkü varyans tahmini χ²(n−1) dağılımından gelir; n küçükse bu dağılımın kuyrukları çok geniştir.

---

## Bölüm 9 – Accuracy / Precision Tartışması

### Neden μ, σ'dan daha hassas tahmin edilebiliyor?

**Teorik açıklama:**

n bağımsız Gaussian gözlemde:
- μ için standart hata: $\text{SE}(\hat{\mu}) = \sigma / \sqrt{n}$ → n arttıkça **√n hızında** azalır
- σ için standart hata: $\text{SE}(\hat{\sigma}) \approx \sigma / \sqrt{2(n-1)}$ → benzer ölçekte ama:

Aradaki fark n büyüdükçe kapanır; ancak **küçük n'de σ tahmini çok daha kararsız** olur. Bu çalışmada n=50 ile her iki parametrenin CI genişliği benzer görünse de, n=5'e geçildiğinde σ'nın **4.5×** büyümesi (μ'nun 3.1×'ine karşı) bu asimetriyi açıkça göstermektedir.

**Corner plot'ta bunu ne gösterir?**
Marjinal posterior genişliklerine bakıldığında σ dağılımının hafif sağa-çarpık (asimetrik) olduğu görülür. Bu asimetri, σ > 0 kısıtından kaynaklanır ve küçük n'de belirginleşir.

---

## Bölüm 10 – Özet ve Sonuç

In [ ]:
print('=' * 65)
print('  TÜM SENARYOLAR – ÖZET')
print('=' * 65)
print(f'  {"Senaryo":<35} {"μ median":>10} {"σ median":>10}')
print(f'  {"-"*55}')
print(f'  {"Temel (n=50, geniş prior)":<35} {mu_med:>10.4f} {sig_med:>10.4f}')
print(f'  {"Dar prior (100<μ<110)":<35} {mu_n_med:>10.4f} {sig_n_med:>10.4f}')
print(f'  {"Az veri (n=5, geniş prior)":<35} {mu5_med:>10.4f} {sig5_med:>10.4f}')
print(f'\n  Gerçek değerler: μ={TRUE_MU}, σ={TRUE_SIGMA}')
print('=' * 65)

### Sonuç

Bu çalışma üç temel bulguyu ortaya koymuştur:

1. **Yeterli veri + uygun prior → başarılı kurtarma:** n=50 ile μ ve σ, gerçek değerlere %2'nin altında hatayla kurtarılmıştır.

2. **Yanlış prior tüm çıkarımı bozar:** Gerçek değeri dışarıda bırakan dar bir prior, σ'nın mantıksız biçimde büyümesine yol açmıştır. Bu, "prior-likelihood çatışması"nın somut göstergesidir.

3. **Veri azaldıkça belirsizlik hızla büyür:** n=5'te güven aralıkları 3–4.5× genişlemiş; bu, Bayesyen çıkarımın veri toplamaya verdiği değeri matematiksel olarak desteklemektedir.

> Bayesyen yöntem belirsizliği gizlemez, ölçer. Bu yüzden astronomi gibi doğrudan deney yapılamayan alanlarda altın standart olarak kabul görmektedir.

---
*YZM212 Makine Öğrenmesi | 2025–2026 Bahar Dönemi*